In [101]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [102]:
# Read words from file
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [103]:
len(words)

32033

In [104]:
# Vocabulary mapping
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [105]:
# Create the dataset of blocks
block_size = 3
X, Y = [], []

for w in words[:5]:
    print(w)
    context = [0] * block_size  # zero context
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(f"{''.join(itos[i] for i in context)} ---> {ch}")
        context = context[1:] + [ix]  # slide context to the right by one

X = torch.tensor(X)
Y = torch.tensor(Y)


emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [106]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

## Embedding

In [ ]:
C = torch.randn((27, 2))  # Embedding table: 27 rows, 2 columns
# thus, X @ C -> row vector of size 2 per character X

In [108]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

## Layer 1

In [109]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [110]:
# concatenate each block in emb along dimension 1 (for input vectors of size 2)
torch.concat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]], 1).shape

torch.Size([32, 6])

In [111]:
# For any block size: torch.unbind removes a dimension (same as above)
torch.concat(torch.unbind(emb, 1), 1).shape  # and concat along the 2nd dimension (dim 1)

torch.Size([32, 6])

In [112]:
emb.view(32, 6).shape

torch.Size([32, 6])

In [113]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [114]:
h.shape

torch.Size([32, 100])

## Layer 2

In [115]:
W2 = torch.randn((100, 27))  # 100 rows, 27 columns
b2 = torch.randn(27)  # converts to 1 row, 27 columns when multiplied

In [116]:
### Layer 2 Calculation
logits = h @ W2 + b2  # (32, 100) @ (100, 27) -> (32, 27)
print(logits.shape)
counts = logits.exp()
prob = counts / counts.sum(1, keepdim=True)  # convert each row into prob. dist. by dividing by row sums (while preserving dim as rows = 1)
print(prob.shape)

torch.Size([32, 27])
torch.Size([32, 27])


In [117]:
prob

tensor([[6.8005e-10, 1.2032e-16, 7.5356e-08, 4.6379e-10, 2.6983e-07, 7.7927e-10,
         2.1722e-07, 4.1931e-12, 7.7284e-04, 6.1532e-18, 7.2129e-05, 1.5817e-10,
         2.8310e-08, 6.4209e-04, 1.1442e-06, 3.3468e-07, 1.3131e-03, 3.3463e-05,
         3.8155e-04, 3.4016e-07, 1.3180e-05, 2.9784e-15, 1.9591e-07, 2.3058e-10,
         8.9003e-05, 9.9668e-01, 5.8709e-15],
        [3.2302e-11, 1.1922e-16, 1.4323e-06, 5.9733e-09, 1.4051e-06, 3.9913e-10,
         8.3283e-08, 6.6795e-12, 1.6156e-03, 1.3511e-17, 6.7639e-06, 1.6039e-10,
         1.9306e-08, 9.1245e-04, 2.1560e-05, 1.1748e-06, 2.3918e-03, 5.5065e-06,
         2.7703e-05, 8.2710e-09, 3.7130e-06, 1.2588e-14, 3.3612e-07, 5.7524e-10,
         2.4974e-04, 9.9476e-01, 1.6300e-12],
        [4.2958e-09, 5.9922e-16, 5.4504e-07, 1.0455e-07, 8.2825e-07, 1.5997e-11,
         5.3700e-08, 7.0300e-13, 2.5358e-02, 8.3164e-16, 7.9603e-04, 1.8393e-07,
         3.7232e-09, 4.9977e-04, 3.8091e-05, 7.9540e-07, 4.9706e-04, 5.0199e-05,
         1.7224e-

## Loss

In [118]:
### Loss calculated as average negative log likelihood (aka. one-hot cross entropy)
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(18.6675)

## Full Model
(but actually Presentable)

In [355]:
block_size = 3
X, Y = [], []

for w in words:
    context = [0] * block_size  # zero context
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]  # slide context to the right by one

X = torch.tensor(X)
Y = torch.tensor(Y)

In [356]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [357]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27, 2), generator=g)
W1 = torch.randn((6, 100), generator=g)
b1 = torch.randn(100, generator=g)
W2 = torch.randn((100, 27), generator=g)
b2 = torch.randn(27, generator=g)
parameters = [C, W1, b1, W2, b2]

In [358]:
sum(p.nelement() for p in parameters)  # total number of parameters

3481

In [359]:
for p in parameters:
    p.requires_grad = True

In [360]:
lre = torch.linspace(-3, 0, 1000)  # from 1e-3 to 1
lrs = 10**lre

In [393]:
lri = []
lossi = []

for i in range(100000):

    # Minibatch construct
    ix = torch.randint(0, X.shape[0], (32,))  # generates 32 random indices in X and Y for minibatch

    # Forward pass
    emb = C[X[ix]]  # (32, 2, 3)
    h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # (32, 100)
    logits = h @ W2 + b2  # (32, 27)
    loss = F.cross_entropy(logits, Y[ix])  # nll = cross-entropy loss (PyTorch fused kernel + bundles softmax)
    # print(loss.item())

    # Backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    # Update
    # lr = lrs[i]
    lr = 0.01
    for p in parameters:
        p.data += -lr * p.grad

    # Track stats
    # lri.append(lre[i])
    # lossi.append(loss.item())

# plt.plot(lri, lossi)  # As we can see, 10^-1 is pretty good, actually.
# loss.item()

In [394]:
emb = C[X]  # (32, 2, 3)
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # (32, 100)
logits = h @ W2 + b2  # (32, 27)
loss = F.cross_entropy(logits, Y)  # nll = cross-entropy loss (PyTorch fused kernel + bundles softmax)
loss

tensor(2.2856, grad_fn=<NllLossBackward0>)

## Data Splits

In [ ]:
# Training split, Dev/Validation split, Test split
# 80%, 10%, 10%

## Sampling

In [395]:
# Finally, we can sample from the Neural Net; note how the results are very similar to the one with pure counts.
g = torch.Generator().manual_seed(2147483647)

for i in range(100):
    out = []
    context = [0] * block_size
    while True:
        emb = C[context]  # (32, 2, 3)
        h = torch.tanh(emb.view(-1, 6) @ W1 + b1)  # (32, 100)
        logits = h @ W2 + b2  # (32, 27)
        counts = logits.exp()  # compute softmax prob distributions
        probs = counts / counts.sum(1, keepdim=True)  

        ix = torch.multinomial(probs, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        context = context[1:] + [ix]
        if ix == 0:
            break
    print(''.join(out))

dex.
maiomlynn.
ile.
kayda.
konist.
tain.
lucan.
kanda.
samiyah.
javhrigotsi.
molie.
kavo.
kerteda.
kaley.
maside.
eniav.
jaylynns.
mhicin.
batahlyn.
kasdr.
ban.
jlen.
awaisan.
jur.
daile.
zam.
deru.
fir.
the.
ika.
bhahbsa.
thrityn.
qexen.
abrica.
hor.
engemalesilm.
javay.
caliy.
oshlbineana.
eviah.
haiah.
avi.
ille.
lad.
idan.
ezhana.
hoama.
morn.
asha.
febalisahr.
joyzeasal.
belane.
nechay.
rakonis.
yana.
isa.
dougen.
luishycaro.
all.
jonnustetorgerraba.
rademkarme.
vhiahdaian.
jayk.
jagh.
kry.
khahlli.
lay.
mari.
prahayd.
endie.
gossin.
mer.
decta.
tiriel.
jedahzyna.
eron.
marvituria.
jovni.
hakirmena.
rena.
raz.
breighacalenne.
mian.
keo.
seahah.
kayrena.
hon.
keika.
soynn.
miciaw.
trazenna.
dimerie.
kediegh.
kiha.
denosi.
kar.
kavmarya.
jamirorleeta.
ledine.
sakin.
